In [2]:
import sys
!{sys.executable} -m pip install git+https://github.com/facebookresearch/segment-anything.git
!{sys.executable} -m pip install opencv-python pillow matplotlib

  Cloning https://github.com/facebookresearch/segment-anything.git to C:\Users\Админ\AppData\Local\Temp\pip-req-build-cuod_2jr
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git 'C:\Users\Админ\AppData\Local\Temp\pip-req-build-cuod_2jr'


In [3]:
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import requests

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
from tqdm.notebook import tqdm

In [6]:
# Принудительное удаление и перезагрузка
dest = Path("sam_vit_h_4b8939.pth")

if dest.exists():
    print(f"Удаляю существующий файл: {dest}")
    dest.unlink()

Удаляю существующий файл: sam_vit_h_4b8939.pth


In [ ]:
# Альтернативный способ загрузки SAM
try:
    print("Загружаю SAM через torch.hub...")
    
    sam = torch.hub.load('facebookresearch/segment-anything', 'sam_vit_h')
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    sam.to(device)
    
    mask_generator = SamAutomaticMaskGenerator(sam)
    
    print("Модель успешно загружена!")

except Exception as e:
    print(f"Ошибка: {e}")

In [ ]:
# скачиваем веса, если ещё не скачаны
# Указываем путь к файлу
dest = Path("sam_vit_h_4b8939.pth")

# Проверяем размер файла
if dest.exists():
    file_size = dest.stat().st_size
    expected_size = 2568786378  # примерно 2.56 GB для sam_vit_h
    
    print(f"Размер файла: {file_size / (1024**3):.2f} GB")
    
    # Если файл слишком маленький или нулевой, удаляем его
    if file_size < expected_size * 0.9:  # Если меньше 90% от ожидаемого размера
        print("Файл повреждён или скачан не полностью. Удаляю...")
        dest.unlink()  # удаляем файл
    else:
        print("Файл выглядит нормально, пробуем загрузить...")
else:
    print("Файл не найден, начинаю загрузку...")

# Загружаем заново, если файла нет
if not dest.exists():
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    
    print("Скачиваю вес модели SAM (~2.5 GB)...")
    print("Это может занять некоторое время...")

    # Скачиваем с проверкой целостности
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(dest, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    
    print(f"\nЗагрузка завершена! Файл сохранён: {dest}")
    print(f"Размер файла: {dest.stat().st_size / (1024**3):.2f} GB")

Размер файла: 0.00 GB
Файл повреждён или скачан не полностью. Удаляю...
Скачиваю вес модели SAM (~2.5 GB)...
Это может занять некоторое время...


Downloading:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

In [6]:
# Загружаем модель SAM
sam = sam_model_registry["vit_h"](checkpoint=dest)
sam.to(device)

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95
)

C:\Users\Админ\Desktop\NN_1\nn_env\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


EOFError: Ran out of input